In [1]:
print("Hello World")

import os

Hello World


In [5]:
# importing the test data
import urllib.request

os.makedirs("data",exist_ok=True)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url,
"data/input.txt")
print("data has been downloaded")

data has been downloaded


In [7]:
# file handling
with open('data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print("length of dataset in characters: ", len(text))
print(text[
    : 1000
])

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hung

In [8]:
# vocabulary creation
# finding the unique elemtnts in it and sorted them out
chars = sorted(list(set(text)))
# this makes a set -> unique and then makes an array and sorts them
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# ### Creating a lookup table, with encode and decode functions

In [9]:
# constructing an an encoder and decoder
stoi = {} # string to index 
itos = {} # index to string
for i,ch in enumerate(chars): # this is us making a dictionary for this to refer later, storing characters with their indexes
    stoi[ch
    ] = i
    itos[i
    ] = ch

# making encode and decode functions
def encode(s):
    encoded_list = []
    for char in s:
        int_val = stoi[char
    ]
        encoded_list.append(int_val)
    return encoded_list

def decode(int_list):
    char_list = []
    for num in int_list:
        char = itos[num
    ]
        char_list.append(char)
    res_string = ''.join(char_list) 
    return res_string

In [15]:
# testing this
encoded_text = encode("Hello I am Mehul")
print(f"Encoded Text: {encoded_text}")
decoded_text = decode(encoded_text)
print(f"Decoded Text: {decoded_text}")

Encoded Text: [20, 43, 50, 50, 53, 1, 21, 1, 39, 51, 1, 25, 43, 46, 59, 50]
Decoded Text: Hello I am Mehul


In [16]:
# much efficient way to write this all
stoi_two = {ch:i for i,ch in enumerate(chars)}
itos_two = {i:ch for i,ch in enumerate(chars)}
encode_two = lambda strs: [stoi_two[c] for c in strs]
decode_two = lambda int_list: "".join([itos_two[int_val] for int_val in int_list])
encoded_text_two = encode_two("Hello I am Mehul")
print(f"Encoded Text: {encoded_text_two}")
decoded_text_two = decode_two(encoded_text_two)
print(f"Decoded Text: {decoded_text_two}")

Encoded Text: [20, 43, 50, 50, 53, 1, 21, 1, 39, 51, 1, 25, 43, 46, 59, 50]
Decoded Text: Hello I am Mehul


NOTE : there are other forms of lookup tables or tokenizers
for eg: Google uses SentencePiece
and openAI has tiktoken

> the better and more vocab dense out tokenizer is: the smaller our token array would be
> this is because we now have more words to fill in
> so smaller sequence -> larger vocabulary
> and larger sequence -> smaller vocablulary 

## Next Up: tokenzie data/input.txt
ok so now we tokenize the Shakespeare text we have
so we will be using PyTorch for this: for the tensor

## Why do we even need a Tensor ?
> tensor: multi dimensional array (0,1,2,3 dimensions)
```python
x = torch.tensor([1, 2, 3])
```
### Why not just use lists ?
- slow, no GPU support and not designed for neural networks

#### Like eg:
- list
    ```python
        result = []
        for i in range(len(a)):
            result.append(a[i] + b[i])
    ```
- tensors (simpler and faster)
    ```python
    a + b
    ```
    > also : ```x = torch.tensor([1,2,3]).to('cuda')``` for GPU support

#### Operations we would do with Tensors
- Embedding lookup
- Matrix multiplication (tensors are regular/rectangular so we can matrix multiply easily/efficiently)
- Softmax
- Backpropagation

#### What does a Tensor look like ?
```python
python_list = [7, 4, 11, 11, 14]
pytorch_tensor = tensor([ 7,  4, 11, 11, 14])
```
```mermaid
text → "hello"
↓
encode → [7, 4, 11, 11, 14]
↓
torch.tensor(...) → tensor([7, 4, 11, 11, 14])
↓
dtype=torch.long → stored as 64-bit integers
```

In [18]:
# importing pytorch
import torch

# now taking the data and making it a tensor
data = torch.tensor(encode(text),dtype=torch.long) # text: is the text we stored from the data/input.txt
print(data.shape,data.type)
print(f"First 1000 characters: {data[:10000]}")

/Users/mehulxyz/dev/personalProjects/smol_LM/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


torch.Size([1115394]) <built-in method type of Tensor object at 0x1351506e0>
First 1000 characters: tensor([18, 47, 56,  ..., 43, 10,  1])


### Some info
| dtype           | meaning             |
| --------------- | ------------------- |
| `torch.float32` | decimal numbers     |
| `torch.int32`   | integers            |
| `torch.long`    | **64-bit integers** |

- since the data is in IDs we use torch.long
> PyTorch embedding layers require integer indices of type ```torch.long```

- what is shape ? : dimensions of the tensor
```python
encode(text) → [12, 4, 5, 9, 2, ...]
data.shape → (N,)
```
> ```N``` = total number of characters in dataset (~1,000,000)

#### For example:
```python
text = "hello"
encode(text) → [7, 4, 11, 11, 14]
data = torch.tensor([7, 4, 11, 11, 14], dtype=torch.long)
print(data.shape)
```
> output: `(5,)`

now we split the data into training and validation set

In [ ]:
val = int(0.9*len(data)) # 90% of the data
train_data = data[:val] # 90% is for the training
test_data = data[val:] # 10% is for the testing
# we do this so that the neural network learns and jsut not memorises the whole dataset
# SAY NO TO OVERFITTING

### Making chunks
- here we can't/won't send the entire dataset because that would be computationally expensive
- another reason we do this is because we want the Transformer to get used to be able to see the input from: as little as `1` to the `block_size` (we are talking about: character size)
- so we make chunks (sample random chunks/sets) from the dataset and feed these chunks at a time.
- length_chunk: `block_size` or `context_length`

In [22]:
# making chunks
block_size = 8
block= train_data[:block_size+1]
print(f"block: {block}")
print(f"Data inside: {decode(block.tolist())}")

block: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
Data inside: First Cit


- the idea is that this tensor will have certain values like above
- this set of values: they follow each other right
- so when we plug this into a transformer then we will be training at each and every position
- for eg: `print(f"block: {block}")` : `block: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])`
- this indicates: for the context of `18`: `47` comes next and for the context of `18` and `47`: `56` comes next
- so now we have 8 (`block_size`) individual examples
- now we visualise this with the example we have

> NOTE: the Transformer will never receive more than the `block_size` as input

### Making chunks

In [23]:
x = train_data[:block_size]
y = train_data[1:block_size+1] # we are moving the window a character ahead: so that it acts as a target
for token in range(block_size):
    context = x[:token+1]
    target = y[token]
    print(f"When the input/context is: {context} then the output/target is:{target}")

When the input/context is: tensor([18]) then the output/target is:47
When the input/context is: tensor([18, 47]) then the output/target is:56
When the input/context is: tensor([18, 47, 56]) then the output/target is:57
When the input/context is: tensor([18, 47, 56, 57]) then the output/target is:58
When the input/context is: tensor([18, 47, 56, 57, 58]) then the output/target is:1
When the input/context is: tensor([18, 47, 56, 57, 58,  1]) then the output/target is:15
When the input/context is: tensor([18, 47, 56, 57, 58,  1, 15]) then the output/target is:47
When the input/context is: tensor([18, 47, 56, 57, 58,  1, 15, 47]) then the output/target is:58


for the sake of efficiency we will be sending these chunks as batches for the sake of efficiency
so the idea is to make chunks to send the whole dataset at once and then to optimise we it we send the small chunks (relative to the dataset we have) in batches to keep the GPUs busy
> since GPUs are good at parallel processing of data
> these chunks will be processed completely independently and don't talk to each other in the process

### Introducing batches
- we will also be using some random values and for that a seed:
  - this is done to pick out random chunks out of the dataset
- here we would be making a tensor of `4 x 8`: so 4 rows (batches) and 8 columns (context size of each chunk)
- each row: chunk
  - how do we do this?: we use `tensor.stack()` to stack chunks in batches of four


In [28]:
torch.manual_seed(1337) # seed: for reproducibility
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else test_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

### Explaination for this
#### ix
```python
ix = torch.randint(len(data) - block_size, (batch_size,))
```
```python
ix = torch.randint(0, len(data) - block_size, (batch_size,)) # same logic
```
- so `torch.randint(low, high, size)`
| Parameter | Meaning                               |
| --------- | ------------------------------------- |
| `low`     | smallest possible integer (inclusive) |
| `high`    | largest possible integer (exclusive)  |
| `size`    | shape of output tensor                |
- this is to sample random positions from the data we have
  - why did we subtract `block_size`: to avoid exceeding it
  - `values ∈ [low, high)`
  - `(batch_size,)`: represents the shape of the tensor
> NOTE:
```python
(batch_size)   # NOT a tuple
(batch_size,)  # tuple with one element
```
> - this gives us `(4,)` : so a 1D tensor with 4 rows
> - `(4,8)` : would be 4 rows and 8 columns


#### x
```python
x = torch.stack([data[i:i+block_size] for i in ix])
```
- similar to what we did for the `context` part
- `torch.stack` : will stack the 1D tensors over each other

#### y
```python
y = torch.stack([data[i+1:i+block_size+1] for i in ix])
```
- similar to the `target` logic in case of the `context`

#### Dry Run for better understanding
```python
data = torch.tensor([10,11,12,13,14,15,16,17,18,19])
```
> - Index:  0  1  2  3  4  5  6  7  8  9
> - Data:  [10 11 12 13 14 15 16 17 18 19]

```python
batch_size = 2
block_size = 3
```
then
```python
ix = torch.randint(0, len(data) - block_size, (batch_size,))
```
- comes out to be : ```ix = torch.randint(0, 7, (2,))```
- eg of what we could get : ```ix = tensor([2, 5])```
then
```python
x = torch.stack([data[i:i+block_size] for i in ix])
```
before stacking :
```python
[
  tensor([12,13,14]),
  tensor([15,16,17])
]
```
after stacking
```
x = tensor([
  [12,13,14],
  [15,16,17]
])
```
then
```python
y = torch.stack([data[i+1:i+block_size+1] for i in ix])
```
finally we get y as
```
y = tensor([
  [13,14,15],
  [16,17,18]
])
```
final output:
```
x =
[
 [12,13,14],
 [15,16,17]
]

y =
[
 [13,14,15],
 [16,17,18]
]
```
so
```
x: [12,13,14]
y: [13,14,15]
```
what the model learns :
| Input context | Target |
| ------------- | ------ |
| [12]          | 13     |
| [12,13]       | 14     |
| [12,13,14]    | 15     |

#### For the final part :
```python
for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
```
- for row 0 :
t=0: context=[12]       → target=13
t=1: context=[12,13]    → target=14
t=2: context=[12,13,14] → target=15

- visualisation :
data:
[10 11 12 13 14 15 16 17 18 19]

Pick i = 2:

x = [12 13 14]
y = [13 14 15]

> this batching helps: ```batch_size``` sequences in parallel

### Now we use a neural network
- simplest option we have: bigram language model
---
- code:
  ```python
    import torch
    import torch.nn as nn # nn: Neural Networks
    from torch.nn import functional as F
    torch. manual_seed (1337) # for reproducibility
    class BigramLanguageModel(nn.Module):
        def __init__(self, vocab_size):
            super ().__init__()
            # each token directly reads off the logits for the next token from a lookup table
            self. token_embedding_table = nn.Embedding(vocab_size, vocab_size)
            # dimensions of the table: vocab_size x vocab_size
            # .Embedding is the wrapper over a tensor of dimensions vocab_size x vocab_size
        def forward(self, idx, targets):
            # idx and targets are both (B,T) tensor of integers
            logits = self. token_embedding_table (idx) # (B,T,C)
            return logits
    m = BigramLanguageModel(vocab_size)
    out = m(xb, yb) # passing the inputs and targets in the model
    print(out.shape)
  ```
  > Output: torch.Size([4, 8, 65])
---
#### Bigram model
> Core idea: The next token depends only on the immediately previous token
> Bigram model = "given current token, predict next token"
- Usually: ```P(w_t | w_1, w_2, ..., w_{t-1})```
- Bigram: ```P(w_t | w_{t-1})``` (the current one depends only on the previous one)
##### Exmaple
```python
    "hello"
```
this becomes
```python
    (h → e)
    (e → l)
    (l → l)
    (l → o)
```
##### Issue with this approach
- very simplistic and cannot capture any long context
  ```"The cat that chased the dog was..."```
    - Bigram only sees the previous word so it struggles
#### What happens inside?:
```python
    self. token_embedding_table = nn.Embedding(vocab_size, vocab_size)
```
- so when we pass ```idx``` in this, say we passed out ```idx: 24```
  - then we *pluck out* the 24 row in the *embedding table*
  - eg for reference:
    ```python
        tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
    ``` 
    -  and so when we are on the
    -  and ```idx:43``` will pluck out the 43rd row from the Embedding table

#### Logits
```python
    logits = self. token_embedding_table (idx) # (B,T,C)
```
- ```PyTorch : torch``` will arrange alln of this in Batch by Time by Channel tensor
  - here : Batch: 4, Time: 8, Channels: 65 (vocab_size)
> ```logits``` : represent the score for the next character in the sequence.
    - we are tryin to predict the next token based on the individual identity of a single token
    > NOTE: the tokens are not talking to each other and here are only concerned about themselves
        - so say we know that I am token number 5 then we know which token comes after token 5 based on the data we have
        > NOTE: we are not passing or considering any history so far

#### Next up Loss function
- so usually we use to measure the quality of predictions with the : *negative log likelihood loss* aka *cross entropy*
##### Cross Entropy
> Cross-entropy measures how wrong your model’s predicted probabilities are compared to the true answer.
- basically a loss function
- eg:
    - vocab_size: 4
    ```python
        Prediction: [0.1, 0.7, 0.1, 0.1]
        Target:      class = 1 
    ```
> $Loss = -log(p_correct)$
- Intuition
| Model confidence              | Loss   |
| ----------------------------- | ------ |
| 0.9 (very confident, correct) | small  |
| 0.5                           | medium |
| 0.01 (very wrong)             | huge   |
```python
    p(correct) = 0.7 → loss ≈ 0.36
    p(correct) = 0.1 → loss ≈ 2.30
```
> $log$ helps in penalising overconfidence and also helps us to work with gradients
- like : ```p = 0.0001``` : loss becomes very large : strong correction signal
- for ```PyTorch``` : ```loss = F.cross_entropy(logits, targets)```
  - internally :
    - applies softmax : convert logits into probabilities
    - apply $-log(p_correct)$
    - average over the batch
##### What we are doing :
```python
    x = input tokens
    y = correct next tokens
    logits → probabilities for next token
    predicted distribution vs actual token
```
so
```python
    Correct token = index 3

    Model output:
    [0.05, 0.10, 0.15, 0.70]

    Loss = -log(0.70) → small → good
```
```python
    increase probability of correct token
    decrease probability of others
```
##### Why are we even using this approach in the first place :
- task : *"predict next token"*
- that is : *multi-class classification problem*
```python
    data → x → model → logits → cross_entropy(logits, y) → loss
    loss.backward() → update weights
```
#### Implementing Cross Entropy
---
- code :
```python
    import torch
    import torch.nn as nn # nn: Neural Networks
    from torch.nn import functional as F
    torch. manual_seed (1337) # for reproducibility
    class BigramLanguageModel(nn.Module):
        def __init__(self, vocab_size):
            super ().__init__()
            # each token directly reads off the logits for the next token from a lookup table
            self. token_embedding_table = nn.Embedding(vocab_size, vocab_size)
            # dimensions of the table: vocab_size x vocab_size
            # .Embedding is the wrapper over a tensor of dimensions vocab_size x vocab_size
        def forward(self, idx, targets):
            # idx and targets are both (B,T) tensor of integers
            logits = self. token_embedding_table (idx) # (B,T,C)
            # loss using cross entropy
            loss = F.cross_entropy(logits,targets)
            return logits, loss
    m = BigramLanguageModel(vocab_size)
    logits, loss = m(xb, yb) # passing the inputs and targets in the model
    print(logits.shape)
```
- output :
```bash
    RuntimeError                              Traceback (most recent call last)
    Cell In[32], line 18
        16         return logits, loss
        17 m = BigramLanguageModel(vocab_size)
    ---> 18 logits, loss = m(xb, yb) # passing the inputs and targets in the model
    ...
```
---
##### Why this happens ? :
- ref : [cross_entropy on PyTorch's documentation page](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
- so first of all we are calling cross_entropy in it's functional form so that we don't have to make a module for it.
- we have to look at how the functions accepts the parameters
- here channels (```C```) are supposed to be the second parameter
- so we need to change our input to more of a ```B x C x T``` instead of a ```B x T x C```
- so we will ```reshape``` our ```logits```

- so
  - we shall use ```view``` to reshape

In [ ]:
import torch
import torch.nn as nn  # nn: Neural Networks
from torch.nn import functional as F

torch. manual_seed (1337) # for reproducibility
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super ().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self. token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        # dimensions of the table: vocab_size x vocab_size
        # .Embedding is the wrapper over a tensor of dimensions vocab_size x vocab_size
    def forward(self, idx, targets):
        # idx and targets are both (B,T) tensor of integers
        logits = self. token_embedding_table (idx) # (B,T,C)
        B, T, C = logits.shape # unpack the logits's dimensions
        # now we make the new logits with the new dimensions we will use
        logits = logits.view(B*T,C)
        # here we are stretching out the first one into a 1D format: B*T and preserving the Channels (C)
        # now we do the same to targets
        targets = targets.view(B*T) # can also do targets.view(-1)
        # loss using cross entropy
        loss = F.cross_entropy(logits,targets)
        return logits, loss
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # passing the inputs and targets in the model
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


### Loss
- so based on our ```vocab_size``` of ```65``, we are expecting a loss of $loss = -ln(1/65) = 4.1743$
- which is far from $4.8786$ which means:
  - initial predictions are not diffuse and they have entropy and so we are predicting/guessing wrong

### Now the genration part
---
- code:
```python
    def generate(self, idx, max_new_tokens) :
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens) :
        # get the predictions
        logits, loss = self(idx)
        # focus only on the last time step
        logits = logitsI:, -1, :] # becomes (B, C)
        # apply softmax to get probabilities
        probs = F.softmax(logits, dim=-1) # (B, C)
        # sample from the distribution
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        # append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
```
- this ```idx``` that we are taking : is the current context of some characters in a batch
- it is also ```B*T```
- what ```generate``` does : it extends this ```B*T``` to ```B*T+1``` then ```+2```, ```+3``` and so on
  - we are generating the next character in all the Batch dimension in the Time dimension
  - this will go on till ```max_new_tokens```
  - then at the last whatever is predicted is concatenated on top of the *previous ```idx```*
    - this concatenation happens along the first dimension which is the time one : shown with ```dim=1``` : this makes the ```B*T+1```
> Job of the ```generate``` is to take ```B*T``` and return ```B*T+1```
- ```logits = logitsI:, -1, :]``` : here we are plucking out the last element in the time dimension and so we get the ```B*C```
- then we convert the logits to probabilites using ```softmax```
- ```multinomial``` is used to sample from those probabilities, ```num_samples=1``` to get ```1``` example from ```PyTorch```
  - this gives us the single prediction of what comes next .
- ```logits, loss = self(idx)``` : here we are not providing targets for the ```forward(self, idx, targets)``` part
  - so we can have ```targets=None``` and then we can have ```loss=None```
- then at last we will do ```print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))```
  - breakdown
    - ```idx=torch.zeros((1, 1), dtype=torch.long``` : this means we wil have a ```1x1```  (batch and time is 1 each) and they are holding a ```0``` in them
    - this is done because ```0``` is used to represent a new line character so it is fair to use it as a starting point.
    - ```dtype=torch.long``` : tensor with integer type
    - we used ```[0]``` because generate works for batches and we needed to pluck out the single dimension : ```(B, T)``` or ```B*T``` ; so we take out the batch dimension with it

#### Combine all that and run

In [39]:
import torch
import torch.nn as nn  # nn: Neural Networks
from torch.nn import functional as F

torch. manual_seed (1337) # for reproducibility
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super ().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self. token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        # dimensions of the table: vocab_size x vocab_size
        # .Embedding is the wrapper over a tensor of dimensions vocab_size x vocab_size
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor of integers
        logits = self. token_embedding_table (idx) # (B,T,C)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape # unpack the logits's dimensions
            # now we make the new logits with the new dimensions we will use
            logits = logits.view(B*T,C)
            # here we are stretching out the first one into a 1D format: B*T and preserving the Channels (C)
            # now we do the same to targets
            targets = targets.view(B*T) # can also do targets.view(-1)
            # loss using cross entropy
            loss = F.cross_entropy(logits,targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb) # passing the inputs and targets in the model
print(logits.shape)
print(loss)
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


#### Result :
- well the output is garbage
- reason: because this is a totally random model
> NOTE: here the ```generate``` function is really elaborate and we are feeding the entire preceeding sequence of characters but with bigram we are just using the previous one character, this would come handy later when we start to cinsider the history.

### Optimizer
- Instead of the ```SDG``` we will be using ```AdamW```
#### Why Optimizer ?
- current approach :
```python
    input → logits → loss
```
- to learn, we need :
```python
    loss → gradients → update weights
```
- rough idea of what we wanna do :
```python
    logits, loss = model(x, y)
    loss.backward()        # compute gradients
    optimizer.step()       # update parameters
    optimizer.zero_grad()  # reset gradients
```
- what an optimizer does :
```python
    parameter = parameter - learning_rate * gradient
```
##### Types :
- SDG (Stochastic Gradient Descent)
    - $θ = θ - lr * ∇θ$
  - this is simple but prone to noise and not for complex use cases
  - can be a bit slow
- Adam (Adaptive Moment Estimation)
  - It tracks :
    - Mean of gradients (momentum)
    - Variance of gradients (adaptive scaling)
- Idea : instead of one learning rate for all parameters, we have different *effective* learning rate per parameter.
  - this makes it :
    - faster in terms of convergence
    - more stable during training
    - needs less tuning compared to tuning for SDG.
- AdamW (Adam + Weight Decay)
  - Simple Adam : weight decay + gradient update
  - AdamW : 
    - $θ = θ - lr * gradient   (Adam step)$
    - $θ = θ - lr * weight_decay * θ   (regularization step)$
  - this helps :
    - AdamW is more stable during training
    - helps with generalization so it is better for Transformers
  - here :
    ```python
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    ```
    - this is a matrix of paramaters : AdamW would be better
  - Implementing AdamW :
    ```python
        optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
    ```   
    - ```m.parameters()``` : ```embedding table (vocab_size × vocab_size)```
    - ```lr=1e-3``` : ```0.001```
    - ```AdamW``` will create an optimizer object which will :
      - track gradients
      - update parameters
    - until now :
      ```python
        logits, loss = m(xb, yb)
      ```
      - here we are only calculating the loss but we are not updating the weights.

In [40]:
# Implementing Optimizer in PyTorch
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)
# for a smaller network like this one we can deal with a higher learning rate

In [58]:
batch_size = 32
for steps in range(100000):
    # get the batch data
    xb, yb = get_batch("train")
    # evaluate the loss
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True) # zero out all the gradients from the previous steps
    loss.backward() # getting the gradients for all the parameters
    optimizer.step() # using those gradients to update the parameters
print(loss.item())

2.535680055618286


In [59]:
print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=400)[0].tolist()))


G cr'THan' m. dene ugrsllt jond,
Lor f beprprt f g our me frow; S:
HELonnd bllovanill!
Wer thivajoses vef y supl.
Th, callis treldilld s meren p le;

BULORCllly p's he tathefet tus:
Toutin;
As t s waiper, wernd, im POUTrewa, icino s LOLy l the choda Isth, who tshes, witheal thosd yo.
Anes Towikese uile wee oze ivends inght listhiurellogathenry;
Kive iondery'dish forothishovir ass,
Wivearshith beau


- next we will move to the bigram.py file
- the objective now is to make the tokens talk to each other
- why ? : to get better outputs
